In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ------ 1. LOAD ------
df = pd.read_csv('/kaggle/input/datasets/zenishadevani/dataset/ai4i2020 (1).csv')
df.drop(columns=['UDI', 'Product ID'], inplace=True)

# ------ 2. ENCODE ------
le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])
df.drop(columns=['Type'], inplace=True)

# ------ 3. SIMULATE EXTERNAL CONTEXT SIGNALS ------
# In real deployment, these would come from weather APIs / V2X logs
# We simulate them as correlated noise to mimic realistic context

n = len(df)

# Ambient humidity (%) — slightly correlated with Air Temp
df['Ambient_Humidity'] = (
    60 + 10 * np.sin(np.linspace(0, 4 * np.pi, n)) +
    np.random.normal(0, 3, n)
).clip(30, 95)

# Factory load factor (0–1) — affects vibration / torque
df['Factory_Load'] = (
    0.5 + 0.3 * np.sin(np.linspace(0, 8 * np.pi, n)) +
    np.random.normal(0, 0.08, n)
).clip(0.1, 1.0)

# Ambient pressure (hPa) — subtle thermal effect
df['Ambient_Pressure_hPa'] = np.random.normal(1013, 5, n)

# Time-of-day proxy (shift pattern: 0=morning, 1=afternoon, 2=night)
df['Shift'] = np.tile([0, 1, 2], n // 3 + 1)[:n]

# Simulated vibration context (external accelerometer, if available)
# Correlate slightly with failure
df['Ext_Vibration'] = (
    0.5 * df['Torque [Nm]'] / df['Torque [Nm]'].max() +
    np.random.normal(0, 0.1, n)
).clip(0, 1)

# ------ 4. DOMAIN FEATURES ------
df['Temp_diff']     = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power_W']       = df['Torque [Nm]'] * (df['Rotational speed [rpm]'] * 2 * np.pi / 60)
df['Wear_x_Torque'] = df['Tool wear [min]'] * df['Torque [Nm]']
df['Stress_proxy']  = df['Torque [Nm]'] / (df['Rotational speed [rpm]'] + 1e-6)

# Physics failure conditions
df['HDF_cond'] = ((df['Temp_diff'] < 8.6) & (df['Rotational speed [rpm]'] < 1380)).astype(int)
df['PWF_cond'] = ((df['Power_W'] < 3500) | (df['Power_W'] > 9000)).astype(int)
df['OSF_cond'] = (df['Wear_x_Torque'] > 11000).astype(int)

# ------ 5. CONTEXTUAL FUSION FEATURES ------
# Effective heat stress = temp_diff adjusted for external humidity
df['Heat_Stress'] = df['Temp_diff'] * (1 + df['Ambient_Humidity'] / 100)

# Load-adjusted torque
df['Load_Torque'] = df['Torque [Nm]'] * df['Factory_Load']

# Wear acceleration under load
df['Wear_Load'] = df['Tool wear [min]'] * df['Factory_Load']

# Combined risk = vibration × wear
df['Vib_Wear_Risk'] = df['Ext_Vibration'] * df['Tool wear [min]']

# ------ 6. ABLATION STUDY SETUP ------
TARGET = 'Machine failure'
FAILURE_COLS = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Set A: Internal only (no external context)
INTERNAL_FEATURES = [
    'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',
    'Torque [Nm]', 'Tool wear [min]', 'Type_enc',
    'Temp_diff', 'Power_W', 'Wear_x_Torque', 'Stress_proxy',
    'HDF_cond', 'PWF_cond', 'OSF_cond'
]

# Set B: Internal + external context
EXTERNAL_FEATURES = [
    'Ambient_Humidity', 'Factory_Load', 'Ambient_Pressure_hPa',
    'Shift', 'Ext_Vibration', 'Heat_Stress', 'Load_Torque',
    'Wear_Load', 'Vib_Wear_Risk'
]
ALL_FEATURES = INTERNAL_FEATURES + EXTERNAL_FEATURES

X_internal = df[INTERNAL_FEATURES]
X_all      = df[ALL_FEATURES]
y          = df[TARGET]

print("Internal features shape:", X_internal.shape)
print("All features shape:     ", X_all.shape)
print(f"\nClass distribution: {y.value_counts().to_dict()}")

# ------ 7. SCALING ------
scaler = StandardScaler()
X_internal_scaled = pd.DataFrame(scaler.fit_transform(X_internal), columns=X_internal.columns)
X_all_scaled      = pd.DataFrame(scaler.fit_transform(X_all),      columns=X_all.columns)

print("\n✅ Feature Engineering V3 (Contextual Fusion) complete.")
print("Use X_internal_scaled vs X_all_scaled for ablation study comparison.")

Internal features shape: (10000, 13)
All features shape:      (10000, 22)

Class distribution: {0: 9661, 1: 339}

✅ Feature Engineering V3 (Contextual Fusion) complete.
Use X_internal_scaled vs X_all_scaled for ablation study comparison.
